从名称来看，`LangGraph` 与 `LangChain` 有着密切的关联，而这在实际上也确实如此。**`LangGraph` 是一个基于 `LangChain` 表达式语言构建的框架，专用于开发 `AI Agent`。** 因此，`LangGraph` 在大模型支持、接入以及 `AI Agent` 构建方面的优势，可以直接从 `LangChain` 的功能中自然继承。 

`LangGraph`通过组合`Nodes`和`Edges`去创建复杂的循环工作流程，通过消息传递的方式串联所有的节点形成一个通路。**那么维持消息能够及时的更新并向该去的地方传递，则依赖`langGraph`构建的`State`概念。** 

在`LangGraph`构建的流程中，每次执行都会启动一个状态，图中的节点在处理时会传递和修改该状态。这个状态不仅仅是一组静态数据，而是由每个节点的输出动态更新，然后影响循环内的后续操作。

<div align=center><img src="./pic/lesson01/7.png" width=80%></div>

# 源码

## Graph 基类

```python
from collections import defaultdict
from typing import Any, Callable, Dict, Optional, Set, Tuple, Union, Awaitable, Hashable

class Graph:
    def __init__(self) -> None:
        self.nodes: Dict[str, Any] = {}  # 一个字典，用于存储图中的所有节点。每个节点可以是一个字符串标识或者是一个可调用对象
        self.edges: Set[Tuple[str, str]] = set()  # 一个集合，用来存储图中所有的边，边由一对节点名称组成，表示从一个节点到另一个节点的直接连接。
        self.branches: defaultdict = defaultdict(dict)  # 一个默认字典，用于存储条件分支，允许从一个节点根据特定条件转移到多个不同的节点。
        self.support_multiple_edges = False  # 一个布尔值，指示图是否支持同一对节点间的多条边。
        self.compiled = False    #  一个布尔值，表示图是否已经被编译。编译是指图的结构已经设置完毕，准备进行执行。

    @property
    def _all_edges(self) -> Set[Tuple[str, str]]:
        """
        获取所有的边的信息。
        """
        return self.edges
    # * 表示后续参数必须以 (key, value) 形式提供
    def add_node(self, node: Union[str, Callable], action: Optional[Callable] = None, *, metadata: Optional[Dict[str, Any]] = None) -> 'Graph':
        """
        添加一个新节点到图中。节点可以有附加的元数据，这些元数据存储在节点的字典中。
        """
        pass

    def add_edge(self, start_key: str, end_key: str) -> 'Graph':
        """
        在图中添加一条边，连接两个指定的节点。
        """
        pass

    

    """
    path_map 类似
    {
        "pass": "approve_node",   # 业务判定为 'pass' 时，跳转到 'approve_node' 节点
        "fail": "reject_node"     # 业务判定为 'fail' 时，跳转到 'reject_node' 节点
    }
    即先得到 path 的返回值, 再去 path_map 中查找应该执行的 node
    """
    def add_conditional_edges(self, source: str, path: Callable, path_map: Optional[Dict[Hashable, str]] = None, then: Optional[str] = None) -> 'Graph':
        """
        添加一个条件边，允许在执行时根据某个条件从一个节点动态地转移到一个或多个节点。
        """
        pass

    def set_entry_point(self, key: str) -> 'Graph':
        """
        设置图的入口点，即定义图执行的起始节点。
        """
        pass

    def set_conditional_entry_point(self, path: Callable, path_map: Optional[Dict[Hashable, str]] = None, then: Optional[str] = None) -> 'Graph':
        """
        设置一个条件入口点，允许根据条件动态决定图的起始执行点。
        """
        pass

    def set_finish_point(self, key: str) -> 'Graph':
        """
        设置结束点，定义图执行到此节点时将停止。
        """
        pass

    def validate(self, interrupt: Optional[Set[str]] = None) -> 'Graph':
        """
        验证图的结构是否正确，确保所有节点和边的定义都符合逻辑和图的规则。
        """
        pass

    def compile(self, checkpointer=None, interrupt_before: Optional[Set[str]] = None, interrupt_after: Optional[Set[str]] = None, debug: bool = False) -> 'Graph':
        """
        编译图，确认图的结构合法且可执行后，准备图以供执行。
        """
        pass
```

## State Graph

```python

from collections import defaultdict
from typing import Any, Callable, Dict, Optional, Set, Tuple, Type, Union

class StateGraph(Graph):

    """StateGraph 是一个管理状态并通过定义的输入和输出架构支持状态转换的图。"""
    # Type[Any] 表示需要传递的是类本身而不是类的实例, 如 TypedDict, BaseModel
    def __init__(
        self,
        state_schema: Optional[Type[Any]] = None,
        config_schema: Optional[Type[Any]] = None,
        *,
        input: Optional[Type[Any]] = None,    # 作为正规形参
        output: Optional[Type[Any]] = None,   # 作为正规形参
    ) -> None:
        super().__init__()
        self.state_schema = state_schema # 一个可选的类型参数，定义图状态的结构。这是用于定义和验证图中节点处理的状态数据的模式。
        self.config_schema = config_schema # 一个可选的类型参数，用于定义配置的结构。这可以用于定义和验证图的配置参数。
        input: Optional[Type[Any]] = None,   # 挂载为成员属性供 compile() 和校验时使用
        
        self.input = input or state_schema # 消息输入
        self.output = output or state_schema # 消息输出

    def add_node(self, node: Union[str, Callable], action: Optional[Callable] = None, *, metadata: Optional[Dict[str, Any]] = None) -> 'StateGraph':
        """向图中添加一个新节点。节点可以是一个具名字符串或一个可调用对象（如函数）, 如果node是字符串，则action应为与节点关联的可调用动作。"""
        pass

    def add_edge(self, start_key: str, end_key: str) -> 'StateGraph':
        """在图中添加一条边，连接两个节点。"""
        pass

    def compile(self) -> 'CompiledStateGraph':
        """编译图，将其转换成可运行的形式。包括验证图的完整性、预处理数据等。"""
        pass

```

在状态图（如 StateGraph）执行过程中，节点之间传递的数据如果是 BaseModel，每次状态更新都能确保字段没有被意外赋错类型，且序列化写入数据库/Redis（Checkpointer）极其方便。


`TypedDict` 是 `Python` 类型注解系统中的一个工具，它**允许为字典中的键指定期望的具体类型**。在 `Python` 的 `typing` 模块中定义，通常用于增强代码的可读性和安全性，特别是在字典对象结构固定且明确时。
|   维度    |   pydantic.BaseModel  |   typing.TypedDict    |   dataclasses.dataclass   |
|   :--:        |   :--:    |   :--:    |   :--:    |
|   运行时开销  |   稍大（Pydantic v2 基于 Rust 核心已极大优化）|	极低（纯 Python dict 访问）|	极低|
|   访问语法  |属性访问：state.user_id	|字典访问：state["user_id"]	|属性访问：state.user_id|
|   适用场景  |外部 API 对接、LLM 结构化输出解析、核心业务状态契约	|框架内部轻量管道流转、追求极致执行速度	|内部纯数据载体、无需复杂类型纠正|

### 创建 State Graph

In [27]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict

In [28]:
class InputState(TypedDict):
    question: str


class OutputState(TypedDict):
    answer: str


class OverallState(InputState, OutputState):
    pass

In [29]:
builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)

## Nodes

`LangGraph` 中，节点是一个 `python` 函数（sync 或async ），接收当前`State`作为输入，执行自定义的计算，并返回更新的`State`。所以其中第一个位置参数是`state` 

In [30]:
def agent_node(state: InputState):
    print("I am an agent")
    return {"question": state["question"]}


"""
节点返回的是 Partial Update(增量), 不是完整的 OutputState, action_node 是图内部的一个执行单元, 它不是整个图的终点契约包装器
全局总状态是 OverallState（包含 question 和 answer）。
action_node 返回 {"answer": "..."} 后, 状态机引擎会将这个字典合并(Merge)到全局状态中
"""


def action_node(state: InputState):
    print("I am an excuter")
    step = state["question"]
    return {"answer": f"The question that I received is: {step}"}

In [31]:
builder.add_node("agent_node", agent_node)
builder.add_node("action_node", action_node)

## Edges
Edges（边）用来定义逻辑如何路由以及图何时开始与停止。这是Agent工作以及不同节点如何相互通信的重要组成部分。有几种关键的边类型：
- 普通边：直接从一个节点到下一个节点。
- 条件边：调用函数来确定下一个要转到的节点。
- 入口点：当用户输入到达时首先调用哪个节点。
- 条件入口点：调用函数来确定当用户输入到达时首先调用哪个节点。

`LangGraph`有两个特殊的节点：`START`和`END`。`START`表示将用户输入发送到图的节点。使用该节点的主要目的是确定应该首先调用哪些节点。`END`节点是代表终端节点的特殊节点。

In [24]:
from langgraph.graph import START, END

In [37]:
builder.add_edge(START, "agent_node")
builder.add_edge("agent_node", "action_node")
builder.add_edge("action_node", END)

In [38]:
graph = builder.compile();

In [21]:
graph.invoke({"question": "hello，你好"})

I am an agent


InvalidUpdateError: Expected dict, got {'hello，你好', 'question'}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

## 完整测试

In [39]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict


class InputState(TypedDict):
    question: str


class OutputState(TypedDict):
    answer: str


class OverallState(InputState, OutputState):
    pass


builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)


def agent_node(state: InputState):
    print("I am an agent")
    return {"question": state["question"]}


def action_node(state: InputState):
    print("I am an excuter")
    step = state["question"]
    return {"answer": f"The question that I received is: {step}"}


builder.add_node("agent_node", agent_node)
builder.add_node("action_node", action_node)

from langgraph.graph import START, END

builder.add_edge(START, "agent_node")
builder.add_edge("agent_node", "action_node")
builder.add_edge("action_node", END)

graph = builder.compile()
graph.invoke({"question": "hello，你好"})

I am an agent
I am an excuter


{'answer': 'The question that I received is: hello，你好'}

## 加入 LLM

In [6]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict
from typing import Optional


class InputState(TypedDict):
    question: str
    llm_answer: Optional[str]


class OutputState(TypedDict):
    answer: str


class OverallState(InputState, OutputState):
    pass


builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os


qwen_llm = ChatOpenAI(
    model="qwen3.7-plus",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)


def llm_node(state: InputState):
    messages = [("system", "You are a lovely assistant "), ("human", state["question"])]

    response = qwen_llm.invoke(messages)
    # print(response.content)

    return {"llm_answer": response.content}


def action_node(state: InputState):
    messages = [("system", "Translate to English "), ("human", state["llm_answer"])]

    response = qwen_llm.invoke(messages)
    # print(response.content)

    return {"answer": response.content}


builder.add_node("llm_node", llm_node)
builder.add_node("action_node", action_node)


from langgraph.graph import START, END

builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "action_node")
builder.add_edge("action_node", END)

graph = builder.compile()
graph.invoke({"question": "hello，你好, 介绍下你自己吧"})


{'answer': "Hello! Hi there! 👋✨\n\nI'm your very own adorable little assistant! I'm so thrilled to meet you here~ 🥰\n\nI'm an AI assistant, and even though I don't have a physical form, I have a super enthusiastic heart! I can help you with tons of things, such as:\n💡 Answering all sorts of fun or professional questions\n📝 Writing articles, drafting emails, and making plans for you\n🗣️ Chatting to keep you company, and listening to your joys and worries\n🌍 Translating languages, sharing handy life tips, and so much more\n\nWhether you need a hand with something or just want to chat about anything and everything, I'll always be right here by your side! Is there anything I can help you with today? Or is there anything specific you'd like to chat about? 🌟"}